# 📊 Análise Exploratória — FiscalAudit AI

Neste notebook vou explorar os dados carregados no banco para entender:
- Volume e distribuição dos dados
- Padrões nas inconsistências detectadas
- Empresas com mais problemas
- Tendências ao longo do tempo

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()

# config dos gráficos
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

In [ ]:
# conectar ao banco
host = os.getenv('DB_HOST', 'localhost')
porta = os.getenv('DB_PORT', '3306')
banco = os.getenv('DB_NAME', 'fiscalaudit')
usuario = os.getenv('DB_USER', 'root')
senha = os.getenv('DB_PASSWORD', '')

url = f"mysql+mysqlconnector://{usuario}:{senha}@{host}:{porta}/{banco}"
engine = create_engine(url)

print("✓ Conectado ao banco")

## 1. Visão Geral dos Dados

In [ ]:
# contagem de registros por tabela
tabelas = ['clientes', 'fornecedores', 'empresas', 'movimentacoes_bancarias',
           'documentos_fiscais', 'contas_financeiras', 'conciliacoes']

contagens = {}
for tabela in tabelas:
    query = f"SELECT COUNT(*) as total FROM {tabela}"
    df = pd.read_sql(query, engine)
    contagens[tabela] = df['total'].iloc[0]

df_contagens = pd.DataFrame(list(contagens.items()), columns=['Tabela', 'Registros'])
df_contagens

In [ ]:
# visualização das contagens
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(df_contagens['Tabela'], df_contagens['Registros'], color='steelblue')
ax.set_xlabel('Quantidade de Registros')
ax.set_title('Volume de Dados por Tabela')
ax.invert_yaxis()

for i, v in enumerate(df_contagens['Registros']):
    ax.text(v + 10, i, str(v), va='center')

plt.tight_layout()
plt.show()

## 2. Análise das Divergências de Pagamento

In [ ]:
# carrega as divergências
query = """
SELECT 
    c.id_conta,
    e.razao_social AS empresa,
    c.tipo_titulo,
    c.valor_original,
    c.valor_pago,
    c.valor_original - c.valor_pago AS diferenca,
    ABS(c.valor_original - c.valor_pago) / c.valor_original * 100 AS perc_diferenca
FROM contas_financeiras c
JOIN empresas e ON c.id_empresa = e.id_empresa
WHERE c.status_pagamento = 'Pago'
  AND c.valor_pago != c.valor_original
"""

df_div = pd.read_sql(query, engine)
print(f"Total de divergências: {len(df_div)}")
df_div.head()

In [ ]:
# estatísticas das divergências
print("Estatísticas das Diferenças (R$):")
print(df_div['diferenca'].describe())
print(f"\nTotal em divergências: R$ {df_div['diferenca'].sum():,.2f}")

In [ ]:
# distribuição dos valores de divergência
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# histograma
axes[0].hist(df_div['diferenca'], bins=15, color='coral', edgecolor='black')
axes[0].set_xlabel('Diferença (R$)')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição das Divergências de Pagamento')

# percentual de diferença
axes[1].hist(df_div['perc_diferenca'], bins=15, color='lightgreen', edgecolor='black')
axes[1].set_xlabel('% Diferença')
axes[1].set_ylabel('Frequência')
axes[1].set_title('Distribuição do Percentual de Divergência')

plt.tight_layout()
plt.show()

In [ ]:
# divergências por tipo de título
div_por_tipo = df_div.groupby('tipo_titulo').agg({
    'id_conta': 'count',
    'diferenca': 'sum'
}).rename(columns={'id_conta': 'quantidade', 'diferenca': 'total_diferenca'})

print(div_por_tipo)

## 3. Análise das Conciliações

In [ ]:
# status das conciliações
query = """
SELECT 
    status_conciliacao,
    COUNT(*) as quantidade,
    SUM(ABS(diferenca_valor)) as total_diferenca
FROM conciliacoes
GROUP BY status_conciliacao
"""

df_conc_status = pd.read_sql(query, engine)
df_conc_status

In [ ]:
# gráfico de pizza do status das conciliações
fig, ax = plt.subplots(figsize=(8, 8))
colors = ['lightgreen', 'salmon']
ax.pie(df_conc_status['quantidade'], labels=df_conc_status['status_conciliacao'],
       autopct='%1.1f%%', startangle=90, colors=colors)
ax.set_title('Status das Conciliações')
plt.show()

In [ ]:
# movimentações não conciliadas
query = """
SELECT 
    tipo_operacao,
    COUNT(*) as quantidade,
    SUM(valor_movimentacao) as total
FROM movimentacoes_bancarias
WHERE conciliada = 0
GROUP BY tipo_operacao
"""

df_mov_nao_conc = pd.read_sql(query, engine)
print("Movimentações Não Conciliadas:")
df_mov_nao_conc

## 4. Análise por Empresa

In [ ]:
# empresas com mais divergências
query = """
SELECT 
    e.razao_social,
    COUNT(*) as qtd_divergencias,
    SUM(ABS(c.valor_original - c.valor_pago)) as total_divergencia
FROM contas_financeiras c
JOIN empresas e ON c.id_empresa = e.id_empresa
WHERE c.status_pagamento = 'Pago'
  AND c.valor_pago != c.valor_original
GROUP BY e.id_empresa, e.razao_social
ORDER BY qtd_divergencias DESC
"""

df_emp_div = pd.read_sql(query, engine)
print("Top 5 Empresas com Mais Divergências:")
df_emp_div.head()

In [ ]:
# gráfico de divergências por empresa
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(range(len(df_emp_div)), df_emp_div['qtd_divergencias'], color='tomato')
ax.set_xticks(range(len(df_emp_div)))
ax.set_xticklabels(df_emp_div['razao_social'], rotation=45, ha='right')
ax.set_ylabel('Quantidade de Divergências')
ax.set_title('Divergências de Pagamento por Empresa')
plt.tight_layout()
plt.show()

## 5. Análise Temporal

In [ ]:
# movimentações ao longo do tempo
query = """
SELECT 
    DATE_FORMAT(data_movimentacao, '%Y-%m') as mes,
    tipo_operacao,
    COUNT(*) as quantidade,
    SUM(valor_movimentacao) as total
FROM movimentacoes_bancarias
GROUP BY mes, tipo_operacao
ORDER BY mes, tipo_operacao
"""

df_temporal = pd.read_sql(query, engine)
df_temporal.head(10)

In [ ]:
# pivot para facilitar a visualização
df_pivot = df_temporal.pivot(index='mes', columns='tipo_operacao', values='total').fillna(0)

fig, ax = plt.subplots(figsize=(14, 6))
df_pivot.plot(kind='bar', ax=ax, color=['green', 'red'])
ax.set_xlabel('Mês')
ax.set_ylabel('Valor Total (R$)')
ax.set_title('Movimentações Bancárias ao Longo do Tempo')
ax.legend(title='Tipo')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Documentos Fiscais

In [ ]:
# distribuição de documentos por tipo e status
query = """
SELECT 
    tipo_documento,
    status_documento,
    COUNT(*) as quantidade
FROM documentos_fiscais
GROUP BY tipo_documento, status_documento
ORDER BY tipo_documento, status_documento
"""

df_docs = pd.read_sql(query, engine)
df_docs

In [ ]:
# pivot dos documentos
df_docs_pivot = df_docs.pivot(index='tipo_documento', columns='status_documento', values='quantidade').fillna(0)

fig, ax = plt.subplots(figsize=(10, 6))
df_docs_pivot.plot(kind='bar', stacked=True, ax=ax, colormap='Set3')
ax.set_xlabel('Tipo de Documento')
ax.set_ylabel('Quantidade')
ax.set_title('Documentos Fiscais por Tipo e Status')
ax.legend(title='Status')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Conclusões

A partir desta análise exploratória, observei:

### Divergências de Pagamento
- **23 contas** apresentam diferença entre valor original e valor pago
- A maioria das divergências está na faixa de 5% a 15%
- Tanto títulos a receber quanto a pagar apresentam divergências

### Conciliações
- ~12% das conciliações estão marcadas como "Inconsistente"
- A maioria das movimentações bancárias (~78%) ainda não está conciliada
- Isso indica que há muitas movimentações sem correspondência no financeiro

### Empresas
- As divergências estão distribuídas entre as empresas
- Algumas empresas concentram mais problemas e podem precisar de atenção especial

### Documentos Fiscais
- ~10% dos documentos estão cancelados ou inutilizados
- A distribuição entre NF-e, NFS-e, NFC-e e CT-e parece balanceada

### Próximos Passos
- Investigar as causas das divergências de pagamento
- Priorizar a conciliação das movimentações bancárias pendentes
- Treinar um modelo de ML para detectar padrões anômalos além das regras fixas

In [ ]:
# resumo final
print("="*60)
print("RESUMO DA ANÁLISE EXPLORATÓRIA")
print("="*60)
print(f"Total de empresas: {contagens['empresas']}")
print(f"Total de movimentações: {contagens['movimentacoes_bancarias']}")
print(f"Total de documentos fiscais: {contagens['documentos_fiscais']}")
print(f"Total de contas financeiras: {contagens['contas_financeiras']}")
print(f"\nInconsistências detectadas:")
print(f"  - Divergências de pagamento: {len(df_div)}")
print(f"  - Conciliações inconsistentes: {df_conc_status[df_conc_status['status_conciliacao']=='Inconsistente']['quantidade'].sum():.0f}")
print(f"  - Movimentações sem conciliação: {df_mov_nao_conc['quantidade'].sum():.0f}")
print("="*60)